# Notebook 04: Preparación de Datos y Análisis Exploratorio (Dataset de Crédito)

Este notebook cubre de forma detallada los conceptos clave para la preparación y preprocesamiento de datos aplicando buenas prácticas en Python y Pandas:
1. **Carga y Exploración Inicial de Datos** (`credito.csv`)
2. **Tratamiento de Valores Duplicados**
3. **Gestión de Valores Faltantes (Missing Values)** (`dropna`, `fillna`)
4. **Estandarización de Datos** (`StandardScaler`)
5. **Detección y Tratamiento de Outliers**
6. **Exploración de Datos (Análisis Univariado y Bivariado)**

## 1. Carga y Exploración Inicial de Datos

Comenzamos importando las librerías necesarias (`pandas`, `numpy`, `matplotlib`, `seaborn`) y cargando el conjunto de datos `credito.csv`.

In [ ]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualizaciones en línea
%matplotlib inline
sns.set_theme(style="whitegrid")

# Carga del dataset de crédito
data = pd.read_csv('credito.csv')

# Visualizar las primeras 10 filas del DataFrame
print("Primeras filas del dataset:")
display(data.head(10))

# Información general sobre dimensiones y tipos de datos
print("\nDimensiones del DataFrame (filas, columnas):")
print(data.shape)

print("\nTipos de datos por columna:")
print(data.dtypes)

print(
    "\nResumen estadístico inicial:")
display(data.describe())

## 2. Tratamiento de Valores Duplicados

Identificamos registros repetidos en el dataset utilizando `duplicated()` y procedemos a eliminarlos de forma segura con `drop_duplicates()`.

In [ ]:
# Identificar filas duplicadas
duplicados = data[data.duplicated()]
print(f"Número de filas duplicadas encontradas: {len(duplicados)}")
if len(duplicados) >:
    display(duplicados)

# Eliminar filas duplicadas conservando la primera ocurrencia
data.drop_duplicates(inplace=True)
print(
    f"Forma del DataFrame después de eliminar duplicados: {data.shape}")

## 3. Gestión de Valores Faltantes (Missing Values)

Para entender cómo manipular valores nulos, simularemos la presencia de datos faltantes (`np.nan`) en algunas columnas y aplicaremos métodos de limpieza (`dropna` y `fillna`).

In [ ]:
# Crear artificialmente valores faltantes con fines educativos
np.random.seed(42)
data.iloc[::3, -1] = np.nan  # Nulos en cada tercer elemento de la última columna ('perfil')
data.iloc[::4, 0] = np.nan   # Nulos en cada cuarto elemento de la primera columna ('ID')

# Comprobar la cantidad de valores nulos por columna
print("Conteo de valores faltantes (NaN) por columna:")
print(data.isna().sum())

# Alternativa A: Eliminar filas que contengan al menos un valor nulo
data_cleaned = data.dropna(axis=0)
print(f"Forma tras aplicar dropna(): {data_cleaned.shape}")

# Alternativa B: Rellenar (imputar) los valores faltantes con fillna()
data_filled = data.copy()
data_filled['perfil'] = data_filled['perfil'].fillna(
    data_filled['perfil'].mean()
)
data_filled['ID'] = data_filled['ID'].fillna(-1)

print(
    "\nValores faltantes después de la imputación con fillna:")
print(data_filled.isna().sum())

## 4. Estandarización de Datos

La estandarización reescala las variables numéricas para que tengan una media de `0` y una desviación estándar de `1`. Esto es indispensable para muchos algoritmos de Machine Learning (como SVM, Redes Neuronales o KNN).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Tomamos el subconjunto sin nulos para realizar la estandarización limpia
df_numeric = data.dropna()

# Instanciar y ajustar el StandardScaler
scaler = StandardScaler()
scaler.fit(df_numeric)

# Transformar los datos numéricos
scaled_array = scaler.transform(df_numeric)
scaled_df = pd.DataFrame(scaled_array, columns=df_numeric.columns)

# Verificar medias y desviaciones estándar resultantes
print("Medias tras estandarizar (deben ser ~0):")
print(scaled_df.mean().round(5))

print(
    "\nDesviaciones estándar tras estandarizar (deben ser 1):")
print(scaled_df.std().round(5))

# Visualización comparativa mediante diagramas de caja (Boxplots)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df_numeric.boxplot(ax=axes[0])
axes[0].set_title('DataFrame Original (Escalas originales)')
axes[0].tick_params(axis='x', rotation=45)

scaled_df.boxplot(ax=axes[1])
axes[1].set_title(
    'DataFrame Estandarizado (StandardScaler)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Detección y Tratamiento de Outliers

Analizamos la presencia de valores atípicos (outliers) utilizando gráficos estadísticos y reglas basadas en la distribución normal (p. ej., umbral de 3 desviaciones estándar).

In [ ]:
# Analizar la distribución de una variable específica, por ejemplo 'ingreso'
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(
    df_numeric['ingreso'], kde=True, bins='auto', color='skyblue'
)
plt.title('Histograma de Ingreso')

plt.subplot(1, 2, 2)
sns.boxplot(x=df_numeric['ingreso'], color='orange')
plt.title('Boxplot de Ingreso')

plt.tight_layout()
plt.show()

# Filtrar outliers utilizando Z-Score (mantener valores dentro de +- 3 desviaciones estándar)
col = 'ingreso'
mean_val = df_numeric[col].mean()
std_val = df_numeric[col].std()

df_no_outliers = df_numeric[
    np.abs(df_numeric[col] - mean_val) <= 3 * std_val
]
print(
    f"Registros originales: {len(df_numeric)} | Registros sin outliers en '{col}': {len(df_no_outliers)}"
)

## 6. Exploración de Datos (Análisis Univariado y Bivariado)

Realizamos un análisis exploratorio detallado para comprender el comportamiento de las variables y sus relaciones en función del `perfil` de crédito.

In [ ]:
# Resumen estadístico completo
print("Estadísticas descriptivas:")
display(df_numeric.describe())

# Análisis agrupado por perfil crediticio
print("\nPromedios agrupados por 'perfil':")
display(df_numeric.groupby('perfil').mean())

# Gráfico de pares (Pairplot) para visualizar relaciones bivariadas clave
subset_cols = ['oferta', 'cuota', 'deuda', 'ingreso', 'perfil']
sns.pairplot(
    df_numeric[subset_cols], hue='perfil', palette='tab10', diag_kind='kde'
)
plt.suptitle(
    'Relaciones Bivariadas según Perfil de Crédito',
    y=1.02
)
plt.show()